# Retinal endothelial Norrin–BRB computational triangulation

This notebook is the public, linear rerun pipeline for three mouse retinal single-cell RNA-sequencing resources:

1. Mouse Retina Cell Atlas (MRCA; Zenodo record 10815031)
2. Furtado et al. / GEO GSE282775
3. Zarkada et al. / GEO GSE175895

It tests whether a frozen Norrin/β-catenin-associated endothelial module covaries with a frozen blood–retinal barrier (BRB) transcriptional module across complementary datasets. It does **not** estimate pathway activity, establish causality, or treat individual cells as independent biological replicates.

## Frozen outcome modules

- Norrin-associated: `Fzd4, Lrp5, Tspan12, Lef1`
- BRB: `Cldn5, Ocln, Tjp1, Mfsd2a, Slc2a1, Abcb1a`
- Junction sensitivity: `Cldn5, Ocln, Tjp1`
- Transport sensitivity: `Mfsd2a, Slc2a1, Abcb1a`

A legacy Norrin definition (`Ndp, Fzd4, Lrp5, Tspan12, Ctnnb1`) is retained **only** as a labelled sensitivity analysis. It never replaces the primary module.

## Validated reference results

- MRCA: 1,436 ECs in 11 sufficiently sampled sequencing cohorts; descriptive random-effects `r = 0.206` (95% CI `0.099–0.309`), `I² = 66.5%`.
- Furtado: 3,017 clean ECs across five pooled genotype libraries; all five within-library Norrin–BRB correlations are positive and essentially unchanged by AV adjustment.
- Zarkada: 683 clean WT ECs define the frozen state map; 527 clean Alk5 ECs are projected into that space; D-tip-like minus S-tip-like scores are positive for Norrin, BRB, junction, and transport modules in 6/6 libraries.

Assertions below stop the run if core counts or headline results drift outside declared tolerances. Reference outputs are under `results/reference/`.


## How to run

From the repository root:

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
python -m pip install --upgrade pip
pip install -r requirements.txt
jupyter lab
```

Open this notebook and run all cells from top to bottom. Set the environment variable `RETINAL_NVU_DOWNLOAD=1` before starting Jupyter to download and extract the public inputs automatically. Otherwise, place the inputs under `data/raw/` as documented in the README.

Expected resources: roughly 8–12 GB of download/storage headroom and at least 16 GB RAM. Raw data are intentionally excluded from version control.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import gc
import gzip
import hashlib
import json
import math
import os
import platform
import sys
import tarfile
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scanpy as sc
import scipy
from IPython.display import display
from scipy import sparse, stats
from scipy.io import mmread
from scipy.stats import pearsonr, rankdata, spearmanr
from sklearn.neighbors import KNeighborsClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("default")

print("Python:", sys.version)
print("Platform:", platform.platform())
for package in ["scanpy", "anndata", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "leidenalg", "igraph"]:
    try:
        print(f"{package}: {version(package)}")
    except Exception:
        print(f"{package}: unavailable")


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists() and (PROJECT_ROOT.parent / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
RESULTS = PROJECT_ROOT / "results" / "generated"
LOGS = RESULTS / "logs"
FIGURES = RESULTS / "figures"
MRCA_DIR = DATA_RAW / "MRCA"
FURTADO_DIR = DATA_RAW / "GSE282775"
ZARKADA_DIR = DATA_RAW / "GSE175895"
MRCA_OUT = RESULTS / "MRCA"
FURTADO_OUT = RESULTS / "FURTADO"
ZARKADA_OUT = RESULTS / "ZARKADA"
FINAL_OUT = RESULTS / "FINAL"

for path in [MRCA_DIR, FURTADO_DIR, ZARKADA_DIR, LOGS, FIGURES, MRCA_OUT, FURTADO_OUT, ZARKADA_OUT, FINAL_OUT]:
    path.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 0
SCORE_CTRL_SIZE = 50
SCORE_N_BINS = 25
MRCA_MIN_EC_PER_COHORT = 20

NORRIN_GENES = ["Fzd4", "Lrp5", "Tspan12", "Lef1"]
BARRIER_GENES = ["Cldn5", "Ocln", "Tjp1", "Mfsd2a", "Slc2a1", "Abcb1a"]
BARRIER_JUNCTION_GENES = ["Cldn5", "Ocln", "Tjp1"]
BARRIER_TRANSPORT_GENES = ["Mfsd2a", "Slc2a1", "Abcb1a"]
LEGACY_NORRIN_GENES = ["Ndp", "Fzd4", "Lrp5", "Tspan12", "Ctnnb1"]
MODULES = {
    "norrin_score": NORRIN_GENES,
    "barrier_score": BARRIER_GENES,
    "junction_score": BARRIER_JUNCTION_GENES,
    "transport_score": BARRIER_TRANSPORT_GENES,
    "legacy_norrin_score": LEGACY_NORRIN_GENES,
}
OUTCOME_GENES = sorted(set(g for genes in MODULES.values() for g in genes))
GENE_ALIASES = {"Abcb1a": ["Abcb1a", "Abcb1"]}

RUN_DOWNLOADS = os.environ.get("RETINAL_NVU_DOWNLOAD", "0") == "1"
print("Project root:", PROJECT_ROOT)
print("Automatic downloads enabled:", RUN_DOWNLOADS)


## Public inputs and provenance

| Resource | Public record | Input used | Analytical role |
|---|---|---|---|
| MRCA | [Zenodo 10815031](https://doi.org/10.5281/zenodo.10815031) | `MRCA- scRNA-seq of the mouse retina - all cells.h5ad` | Within-cohort correlations across source-annotated ECs |
| Furtado | [GEO GSE282775](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE282775) | Five processed 10x count matrices | Within-library Norrin–BRB association and AV adjustment |
| Zarkada | [GEO GSE175895](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE175895) | Three WT and three Alk5 processed 10x count matrices | Frozen WT state mapping and Alk5 projection |

MRCA is an integrated atlas; its 11 eligible sequencing/preparation cohorts are not equivalent to 11 independent animals. The Furtado genotype libraries are pooled preparations. Zarkada libraries have unequal state-specific cell counts, including a small Alk5 E7 comparison. All cell-level summaries are descriptive.


In [ ]:
MRCA_URL = "https://zenodo.org/records/10815031/files/MRCA-%20scRNA-seq%20of%20the%20mouse%20retina%20-%20all%20cells.h5ad?download=1"
MRCA_MD5 = "cbbd7cc9802330a7002946757b995f26"
GEO_URLS = {
    "GSE282775": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE282nnn/GSE282775/suppl/GSE282775_RAW.tar",
    "GSE175895": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE175nnn/GSE175895/suppl/GSE175895_RAW.tar",
}

def digest(path, algorithm="sha256", chunk_size=1024 * 1024):
    h = hashlib.new(algorithm)
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def stream_download(url, destination, expected_md5=None):
    destination = Path(destination)
    if destination.exists():
        if expected_md5 and digest(destination, "md5") != expected_md5:
            raise ValueError(f"Checksum mismatch for existing file: {destination}")
        print("Already present:", destination)
        return destination
    partial = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with partial.open("wb") as handle, tqdm(total=total, unit="B", unit_scale=True, desc=destination.name) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    bar.update(len(chunk))
    if expected_md5 and digest(partial, "md5") != expected_md5:
        raise ValueError(f"Checksum mismatch after download: {partial}")
    partial.replace(destination)
    return destination

def safe_extract_tar(archive, destination):
    archive, destination = Path(archive), Path(destination)
    marker = destination / f".{archive.name}.extracted.ok"
    if marker.exists():
        print("Already extracted:", archive.name)
        return
    base = destination.resolve()
    with tarfile.open(archive, "r") as bundle:
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if os.path.commonpath([str(base), str(target)]) != str(base):
                raise ValueError(f"Unsafe path in archive: {member.name}")
        bundle.extractall(destination, filter="data")
    marker.write_text("ok\n", encoding="utf-8")

if RUN_DOWNLOADS:
    stream_download(MRCA_URL, MRCA_DIR / "MRCA_all_cells.h5ad", expected_md5=MRCA_MD5)
    for accession, url in GEO_URLS.items():
        folder = FURTADO_DIR if accession == "GSE282775" else ZARKADA_DIR
        archive = stream_download(url, folder / f"{accession}_RAW.tar")
        safe_extract_tar(archive, folder)
else:
    print("Skipping downloads. Set RETINAL_NVU_DOWNLOAD=1 before launching Jupyter to enable them.")


## Shared loaders, scoring, and statistical helpers


In [ ]:
def _is_gzip_file(path):
    with Path(path).open("rb") as handle:
        return handle.read(2) == b"\x1f\x8b"

def _open_text_auto(path):
    return gzip.open(path, "rt") if _is_gzip_file(path) else Path(path).open("rt")

def discover_geo_prefixes(folder):
    prefixes = []
    for path in sorted(Path(folder).glob("*.barcodes.tsv.gz")):
        prefixes.append(path.name.replace(".barcodes.tsv.gz", ""))
    for path in sorted(Path(folder).glob("*_barcodes.tsv.gz")):
        prefix = path.name.replace("_barcodes.tsv.gz", "")
        if prefix not in prefixes:
            prefixes.append(prefix)
    return prefixes

def _find_triplet_file(folder, prefix, kind):
    suffixes = [
        f"{prefix}.{kind}.tsv.gz" if kind != "matrix" else f"{prefix}.matrix.mtx.gz",
        f"{prefix}_{kind}.tsv.gz" if kind != "matrix" else f"{prefix}_matrix.mtx.gz",
    ]
    for name in suffixes:
        path = Path(folder) / name
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {kind} file for {prefix} in {folder}")

def read_geo_10x_triplet(folder, prefix):
    barcode_path = _find_triplet_file(folder, prefix, "barcodes")
    feature_path = _find_triplet_file(folder, prefix, "features")
    matrix_path = _find_triplet_file(folder, prefix, "matrix")
    with _open_text_auto(barcode_path) as handle:
        barcodes = pd.read_csv(handle, sep="\t", header=None)
    with _open_text_auto(feature_path) as handle:
        features = pd.read_csv(handle, sep="\t", header=None)
    opener = gzip.open if _is_gzip_file(matrix_path) else open
    with opener(matrix_path, "rb") as handle:
        X = mmread(handle).tocsr().T
    if X.shape != (len(barcodes), len(features)):
        raise ValueError(f"Dimension mismatch for {prefix}: {X.shape}")
    symbols = features.iloc[:, 1].astype(str).to_numpy() if features.shape[1] >= 2 else features.iloc[:, 0].astype(str).to_numpy()
    obs = pd.DataFrame(index=[f"{prefix}:{x}" for x in barcodes.iloc[:, 0].astype(str)])
    obs["sample"] = prefix
    var = pd.DataFrame(index=pd.Index(symbols, name="gene_symbol"))
    var["gene_id"] = features.iloc[:, 0].astype(str).to_numpy()
    result = ad.AnnData(X=X, obs=obs, var=var)
    result.var_names_make_unique()
    return result

def resolve_gene(adata, symbol):
    candidates = [symbol] + GENE_ALIASES.get(symbol, [])
    for candidate in candidates:
        if candidate in adata.var_names:
            return candidate
    if "gene_symbols" in adata.var.columns:
        symbols = adata.var["gene_symbols"].astype(str)
        for candidate in candidates:
            hits = adata.var_names[symbols.eq(candidate)]
            if len(hits) == 1:
                return hits[0]
            if len(hits) > 1:
                raise ValueError(f"Multiple features map to {candidate}: {hits.tolist()}")
    return None

def gene_vector(adata, gene, layer=None):
    feature = resolve_gene(adata, gene)
    if feature is None:
        raise KeyError(gene)
    X = adata[:, feature].layers[layer] if layer else adata[:, feature].X
    return np.asarray(X.toarray()).ravel() if sparse.issparse(X) else np.asarray(X).ravel()

def marker_hit_count(adata, genes, layer=None):
    missing = [gene for gene in genes if resolve_gene(adata, gene) is None]
    if missing:
        raise KeyError(f"Missing marker genes: {missing}")
    return np.vstack([gene_vector(adata, gene, layer=layer) > 0 for gene in genes]).T.sum(axis=1)

def add_module_scores(adata):
    for score_name, genes in MODULES.items():
        resolved = [resolve_gene(adata, gene) for gene in genes]
        missing = [gene for gene, feature in zip(genes, resolved) if feature is None]
        resolved = [feature for feature in resolved if feature is not None]
        minimum = 3 if score_name in {"norrin_score", "legacy_norrin_score"} else 2
        if len(resolved) < minimum:
            raise ValueError(f"{score_name}: too many missing genes: {missing}")
        sc.tl.score_genes(
            adata,
            gene_list=resolved,
            score_name=score_name,
            ctrl_size=SCORE_CTRL_SIZE,
            n_bins=SCORE_N_BINS,
            random_state=RANDOM_SEED,
            use_raw=False,
        )
    return adata

def dl_pool_correlations(frame, r_col, n_col="n_EC"):
    data = frame[[r_col, n_col]].dropna().copy()
    data = data[data[n_col] > 3].copy()
    data["z"] = np.arctanh(np.clip(data[r_col], -0.999999, 0.999999))
    data["var_z"] = 1 / (data[n_col] - 3)
    data["w_fixed"] = 1 / data["var_z"]
    z_fixed = np.sum(data["w_fixed"] * data["z"]) / np.sum(data["w_fixed"])
    Q = np.sum(data["w_fixed"] * (data["z"] - z_fixed) ** 2)
    k = len(data)
    qdf = k - 1
    C = np.sum(data["w_fixed"]) - np.sum(data["w_fixed"] ** 2) / np.sum(data["w_fixed"])
    tau2 = max(0, (Q - qdf) / C) if C > 0 else 0
    data["w_random"] = 1 / (data["var_z"] + tau2)
    z_random = np.sum(data["w_random"] * data["z"]) / np.sum(data["w_random"])
    se = math.sqrt(1 / np.sum(data["w_random"]))
    return {
        "k": int(k),
        "pooled_r": float(np.tanh(z_random)),
        "ci_low": float(np.tanh(z_random - 1.96 * se)),
        "ci_high": float(np.tanh(z_random + 1.96 * se)),
        "p": float(2 * stats.norm.sf(abs(z_random / se))),
        "Q": float(Q),
        "Q_df": int(qdf),
        "Q_p": float(stats.chi2.sf(Q, qdf)),
        "tau2_z": float(tau2),
        "I2_percent": float(max(0, (Q - qdf) / Q * 100)) if Q > 0 else 0.0,
    }

def partial_spearman(x, y, covariate):
    xr, yr, zr = rankdata(x), rankdata(y), rankdata(covariate)
    design = np.column_stack([np.ones(len(zr)), zr])
    x_resid = xr - design @ np.linalg.lstsq(design, xr, rcond=None)[0]
    y_resid = yr - design @ np.linalg.lstsq(design, yr, rcond=None)[0]
    return float(pearsonr(x_resid, y_resid).statistic)

def write_input_manifest(paths):
    rows = []
    for path in paths:
        path = Path(path)
        if path.exists():
            rows.append({"path": str(path.relative_to(PROJECT_ROOT)), "size_bytes": path.stat().st_size, "sha256": digest(path)})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(LOGS / "input_manifest_sha256.csv", index=False)
    return manifest


## A. Mouse Retina Cell Atlas

ECs are selected using the source-provided `majorclass == "Endothelial"` annotation. The deposited expression matrix is used without re-normalization. Correlations are computed separately within `sampleid` cohorts having at least 20 ECs, then summarized descriptively on Fisher-z values using a DerSimonian–Laird random-effects model.


In [ ]:
mrca_path = MRCA_DIR / "MRCA_all_cells.h5ad"
if not mrca_path.exists():
    raise FileNotFoundError(f"Missing {mrca_path}. Enable downloads or follow README data setup.")

mrca_backed = sc.read_h5ad(mrca_path, backed="r")
required_obs = {"majorclass", "sampleid", "reference", "age", "donor_id", "accession"}
missing_obs = required_obs.difference(mrca_backed.obs.columns)
if missing_obs:
    raise KeyError(f"MRCA metadata columns missing: {sorted(missing_obs)}")

ec_mask = mrca_backed.obs["majorclass"].astype(str).eq("Endothelial")
mrca_ec = mrca_backed[ec_mask, :].to_memory()
add_module_scores(mrca_ec)

ec_counts = mrca_ec.obs["sampleid"].value_counts()
eligible_samples = ec_counts[ec_counts >= MRCA_MIN_EC_PER_COHORT].index.tolist()
rows = []
for sample in eligible_samples:
    frame = mrca_ec.obs.loc[mrca_ec.obs["sampleid"].astype(str) == str(sample)]
    row = {
        "sampleid": sample,
        "n_EC": len(frame),
        "reference": str(frame["reference"].iloc[0]),
        "age": str(frame["age"].iloc[0]),
    }
    for label, x, y in [
        ("primary", "norrin_score", "barrier_score"),
        ("junction", "norrin_score", "junction_score"),
        ("transport", "norrin_score", "transport_score"),
        ("legacy", "legacy_norrin_score", "barrier_score"),
    ]:
        result = pearsonr(frame[x].astype(float), frame[y].astype(float))
        row[f"r_{label}"] = float(result.statistic)
        row[f"p_{label}"] = float(result.pvalue)
    rows.append(row)

mrca_cohort_results = pd.DataFrame(rows)
mrca_meta_results = {
    "PRIMARY_NORRIN_vs_BRB": dl_pool_correlations(mrca_cohort_results, "r_primary"),
    "PRIMARY_NORRIN_vs_JUNCTION": dl_pool_correlations(mrca_cohort_results, "r_junction"),
    "PRIMARY_NORRIN_vs_TRANSPORT": dl_pool_correlations(mrca_cohort_results, "r_transport"),
    "LEGACY_NORRIN_vs_BRB": dl_pool_correlations(mrca_cohort_results, "r_legacy"),
}

primary = mrca_meta_results["PRIMARY_NORRIN_vs_BRB"]
assert len(eligible_samples) == 11, f"Expected 11 eligible MRCA cohorts, observed {len(eligible_samples)}"
assert int(mrca_cohort_results["n_EC"].sum()) == 1436
assert np.isclose(primary["pooled_r"], 0.2060097315, atol=1e-6)
assert np.isclose(primary["ci_low"], 0.0986720300, atol=1e-6)
assert np.isclose(primary["ci_high"], 0.3086088586, atol=1e-6)
assert np.isclose(primary["I2_percent"], 66.54151894, atol=1e-5)

mrca_cohort_results.to_csv(MRCA_OUT / "MRCA_11cohort_correlations.csv", index=False)
(MRCA_OUT / "MRCA_random_effects_summary.json").write_text(json.dumps(mrca_meta_results, indent=2), encoding="utf-8")
display(mrca_cohort_results.sort_values("n_EC", ascending=False).reset_index(drop=True))
display(pd.DataFrame(mrca_meta_results).T)


## B. Furtado / GSE282775

The five libraries are pooled genotype preparations, not replicated animals. Candidate ECs require expression of at least 3/6 independent identity markers. Two control libraries are clustered using non-outcome highly variable genes to identify a 13-cell mural/pericyte-contaminated cluster. Perturbation libraries use a frozen marker rule (`<2` of five mural genes). A continuous AV axis is computed from markers that do not overlap the Norrin or BRB outcome modules. Raw and AV-adjusted within-library Spearman associations are descriptive.


In [ ]:
FURTADO_LIBRARIES = [
    ("GSM8650444_Ntn1flfl", "Ntn1flfl", "control"),
    ("GSM8650445_Ntn1iKO", "Ntn1iKO", "perturbation"),
    ("GSM8650446_Unc5bflfl", "Unc5bflfl", "control"),
    ("GSM8650447_Unc5biECKO", "Unc5biECKO", "perturbation"),
    ("GSM8650448_Unc5biECKO_Ctnnb1flex3", "Unc5biECKO_Ctnnb1flex3", "perturbation"),
]
FURTADO_EC_ID_MARKERS = ["Pecam1", "Kdr", "Cdh5", "Esam", "Emcn", "Tek"]
MURAL_QC_GENES = ["Rgs5", "Pdgfrb", "Cspg4", "Abcc9", "Kcnj8"]
ARTERIAL_ZONE_GENES = ["Gja5", "Bmx", "Efnb2", "Sox17"]
CAPILLARY_ZONE_GENES = ["Rgcc", "Car4"]
VENOUS_ZONE_GENES = ["Nr2f2", "Slc38a5"]

def process_furtado_library(prefix, genotype, group):
    raw = read_geo_10x_triplet(FURTADO_DIR, prefix)
    keep = marker_hit_count(raw, FURTADO_EC_ID_MARKERS) >= 3
    result = raw[keep].copy()
    result.obs["sample"] = prefix
    result.obs["genotype"] = genotype
    result.obs["group"] = group
    result.layers["counts"] = result.X.copy()
    sc.pp.normalize_total(result, target_sum=1e4)
    sc.pp.log1p(result)
    add_module_scores(result)
    del raw
    gc.collect()
    return result

furtado = {prefix: process_furtado_library(prefix, genotype, group) for prefix, genotype, group in FURTADO_LIBRARIES}
expected_candidates = {
    "GSM8650444_Ntn1flfl": 643,
    "GSM8650445_Ntn1iKO": 624,
    "GSM8650446_Unc5bflfl": 94,
    "GSM8650447_Unc5biECKO": 124,
    "GSM8650448_Unc5biECKO_Ctnnb1flex3": 1567,
}
observed_candidates = {sample: obj.n_obs for sample, obj in furtado.items()}
assert observed_candidates == expected_candidates, (observed_candidates, expected_candidates)

control_names = ["GSM8650444_Ntn1flfl", "GSM8650446_Unc5bflfl"]
furtado_controls = ad.concat([furtado[name] for name in control_names], join="inner", merge="same", index_unique=None)
controls_cluster = furtado_controls.copy()
sc.pp.highly_variable_genes(controls_cluster, n_top_genes=2000, flavor="seurat", batch_key="sample")
for gene in OUTCOME_GENES:
    if gene in controls_cluster.var_names:
        controls_cluster.var.loc[gene, "highly_variable"] = False
sc.pp.scale(controls_cluster, zero_center=True, max_value=10)
sc.tl.pca(controls_cluster, n_comps=30, use_highly_variable=True, random_state=RANDOM_SEED)
sc.pp.neighbors(controls_cluster, n_neighbors=15, n_pcs=20, random_state=RANDOM_SEED)
sc.tl.leiden(
    controls_cluster,
    resolution=0.6,
    key_added="leiden_independent",
    random_state=RANDOM_SEED,
    flavor="leidenalg",
    directed=True,
    n_iterations=-1,
)
expected_cluster_sizes = {"0": 146, "1": 132, "2": 116, "3": 113, "4": 90, "5": 73, "6": 54, "7": 13}
observed_cluster_sizes = controls_cluster.obs["leiden_independent"].astype(str).value_counts().sort_index().to_dict()
assert observed_cluster_sizes == expected_cluster_sizes, (observed_cluster_sizes, expected_cluster_sizes)
furtado_controls.obs["leiden_independent"] = controls_cluster.obs["leiden_independent"].reindex(furtado_controls.obs_names).astype(str)

independent_zone_markers = [
    "Gja5", "Efnb2", "Bmx", "Sox17", "Cthrc1",
    "Rgcc", "Car4", "Aplnr",
    "Nr2f2", "Ephb4", "Slc38a5",
    "Esm1", "Apln", "Kcne3",
    "Mki67", "Top2a", "Ccnb2",
]
marker_rows = []
for cluster in sorted(furtado_controls.obs["leiden_independent"].unique(), key=int):
    subset = furtado_controls[furtado_controls.obs["leiden_independent"].eq(cluster)]
    row = {"cluster": cluster, "n_cells": subset.n_obs}
    for gene in independent_zone_markers:
        if gene in subset.var_names:
            values = gene_vector(subset, gene)
            row[f"{gene}_mean"] = float(np.mean(values))
            row[f"{gene}_pct_positive"] = float(100 * np.mean(values > 0))
    marker_rows.append(row)
furtado_control_marker_evidence = pd.DataFrame(marker_rows)
furtado_control_marker_evidence.to_csv(FURTADO_OUT / "Furtado_control_independent_marker_evidence.csv", index=False)

contamination_panels = {
    "mural": MURAL_QC_GENES,
    "photoreceptor": ["Rho", "Pde6g", "Rcvrn", "Gngt1", "Sag"],
    "neuronal": ["Syt1", "Snap25", "Tubb3", "Elavl3", "Rbfox3"],
    "immune": ["Ptprc", "Aif1", "C1qa", "C1qb"],
}
qc_rows = []
for cluster in sorted(furtado_controls.obs["leiden_independent"].unique(), key=int):
    subset = furtado_controls[furtado_controls.obs["leiden_independent"].eq(cluster)]
    row = {"cluster": cluster, "n_cells": subset.n_obs}
    for panel, genes in contamination_panels.items():
        present = [g for g in genes if g in subset.var_names]
        hits = marker_hit_count(subset, present)
        row[f"{panel}_median_hits"] = float(np.median(hits))
        row[f"{panel}_pct_ge2"] = float(100 * np.mean(hits >= 2))
    qc_rows.append(row)
furtado_control_cluster_qc = pd.DataFrame(qc_rows)
furtado_control_cluster_qc.to_csv(FURTADO_OUT / "Furtado_control_cluster_QC.csv", index=False)

furtado_controls_clean = furtado_controls[~furtado_controls.obs["leiden_independent"].eq("7")].copy()
assert furtado_controls_clean.n_obs == 724
display(furtado_control_cluster_qc)
display(furtado_control_marker_evidence)


In [ ]:
def add_av_axis(adata):
    genes = ARTERIAL_ZONE_GENES + CAPILLARY_ZONE_GENES + VENOUS_ZONE_GENES
    missing = [gene for gene in genes if gene not in adata.var_names]
    if missing:
        raise KeyError(f"Missing AV genes: {missing}")
    expression = adata[:, genes].X
    expression = expression.toarray() if sparse.issparse(expression) else np.asarray(expression)
    frame = pd.DataFrame(expression, index=adata.obs_names, columns=genes)
    sd = frame.std(axis=0, ddof=0).replace(0, 1)
    z = ((frame - frame.mean(axis=0)) / sd).clip(-3, 3)
    adata.obs["arterial_zone_score"] = z[ARTERIAL_ZONE_GENES].mean(axis=1)
    adata.obs["capillary_zone_score"] = z[CAPILLARY_ZONE_GENES].mean(axis=1)
    adata.obs["venous_zone_score"] = z[VENOUS_ZONE_GENES].mean(axis=1)
    adata.obs["AV_axis"] = adata.obs["arterial_zone_score"] - adata.obs["venous_zone_score"]
    return adata

control_rows = []
for sample in control_names:
    sample_obj = furtado_controls_clean[furtado_controls_clean.obs["sample"].astype(str).eq(sample)].copy()
    add_av_axis(sample_obj)
    for column in ["arterial_zone_score", "capillary_zone_score", "venous_zone_score", "AV_axis"]:
        furtado_controls_clean.obs.loc[sample_obj.obs_names, column] = sample_obj.obs[column]
    raw = float(spearmanr(sample_obj.obs["norrin_score"], sample_obj.obs["barrier_score"]).statistic)
    adjusted = partial_spearman(sample_obj.obs["norrin_score"], sample_obj.obs["barrier_score"], sample_obj.obs["AV_axis"])
    control_rows.append({
        "sample": sample,
        "n_cells": sample_obj.n_obs,
        "analysis_arm": "control",
        "raw_Norrin_vs_BRB_rho": raw,
        "AV_adjusted_partial_rho": adjusted,
        "change_after_AV_adjustment": adjusted - raw,
    })

perturbation_rows, perturbation_qc_rows, perturbation_clean = [], [], {}
for sample in [name for name in furtado if name not in control_names]:
    obj = furtado[sample]
    mural_n = marker_hit_count(obj, MURAL_QC_GENES, layer="counts")
    keep = mural_n < 2
    clean = obj[keep].copy()
    add_av_axis(clean)
    raw = float(spearmanr(clean.obs["norrin_score"], clean.obs["barrier_score"]).statistic)
    adjusted = partial_spearman(clean.obs["norrin_score"], clean.obs["barrier_score"], clean.obs["AV_axis"])
    perturbation_qc_rows.append({
        "sample": sample,
        "cells_before": obj.n_obs,
        "mural_excluded": int((~keep).sum()),
        "cells_after": clean.n_obs,
    })
    perturbation_rows.append({
        "sample": sample,
        "n_cells": clean.n_obs,
        "analysis_arm": "perturbation",
        "raw_Norrin_vs_BRB_rho": raw,
        "AV_adjusted_partial_rho": adjusted,
        "change_after_AV_adjustment": adjusted - raw,
    })
    perturbation_clean[sample] = clean

furtado_5library_summary = pd.DataFrame(control_rows + perturbation_rows)
expected_clean = {
    "GSM8650444_Ntn1flfl": 635,
    "GSM8650446_Unc5bflfl": 89,
    "GSM8650445_Ntn1iKO": 618,
    "GSM8650447_Unc5biECKO": 122,
    "GSM8650448_Unc5biECKO_Ctnnb1flex3": 1553,
}
assert furtado_5library_summary.set_index("sample")["n_cells"].to_dict() == expected_clean
assert int(furtado_5library_summary["n_cells"].sum()) == 3017
expected_raw = np.array([0.214, 0.178, 0.236, 0.122, 0.236])
assert np.allclose(furtado_5library_summary["raw_Norrin_vs_BRB_rho"].to_numpy(), expected_raw, atol=0.0006)
assert (furtado_5library_summary["raw_Norrin_vs_BRB_rho"] > 0).all()
assert (furtado_5library_summary["change_after_AV_adjustment"].abs() < 0.005).all()

furtado_5library_summary.to_csv(FURTADO_OUT / "Furtado_5library_Norrin_BRB_summary.csv", index=False)
pd.DataFrame(perturbation_qc_rows).to_csv(FURTADO_OUT / "Furtado_perturbation_QC.csv", index=False)
display(furtado_5library_summary.round(3))


## C. Zarkada / GSE175895

All six libraries use frozen barcode, mitochondrial, EC-identity, and contamination thresholds. WT libraries alone define an outcome-gene-excluded PCA/Leiden state space. Strongly photoreceptor-contaminated cells are removed using an empirical rule frozen from the validated analysis. Alk5 cells are never reclustered: they are projected onto the WT PCA space with a distance-weighted 15-nearest-neighbour classifier, and cells with confidence below 0.50 remain unassigned.


In [ ]:
ZARKADA_LIBRARIES = [
    ("GSM5350878_P6_WT", "P6", "WT"),
    ("GSM5350880_P10_WT_D7", "P10", "WT"),
    ("GSM5350881_P10_WT_C7", "P10", "WT"),
    ("GSM5350879_P6_Alk5", "P6", "Alk5"),
    ("GSM5350882_P10_Alk5_B7", "P10", "Alk5"),
    ("GSM5350883_P10_Alk5_E7", "P10", "Alk5"),
]
ZARKADA_EC_ID_MARKERS = ["Pecam1", "Kdr", "Cdh5", "Esam", "Emcn", "Tek"]
IMMUNE_QC_GENES = ["Ptprc", "Aif1", "C1qa"]

def process_zarkada_library(prefix, age, genotype):
    raw = read_geo_10x_triplet(ZARKADA_DIR, prefix)
    X = raw.X
    total_umi = np.asarray(X.sum(axis=1)).ravel()
    detected_genes = X.getnnz(axis=1) if sparse.issparse(X) else np.count_nonzero(X, axis=1)
    count_keep = (total_umi >= 500) & (detected_genes >= 100) & (total_umi <= 10000)
    candidate = raw[count_keep].copy()
    mt_mask = np.array([str(g).upper().startswith("MT-") for g in candidate.var_names])
    candidate_total = np.asarray(candidate.X.sum(axis=1)).ravel()
    mt_counts = np.asarray(candidate[:, mt_mask].X.sum(axis=1)).ravel()
    mt_keep = (100 * mt_counts / candidate_total) <= 15
    cells = candidate[mt_keep].copy()
    ec_keep = marker_hit_count(cells, ZARKADA_EC_ID_MARKERS) >= 3
    ec = cells[ec_keep].copy()
    mural_n = marker_hit_count(ec, MURAL_QC_GENES)
    immune_n = marker_hit_count(ec, IMMUNE_QC_GENES)
    clean_keep = (mural_n < 2) & (immune_n < 2)
    clean = ec[clean_keep].copy()
    clean.obs["sample"] = prefix
    clean.obs["age"] = age
    clean.obs["genotype"] = genotype
    clean.layers["counts"] = clean.X.copy()
    sc.pp.normalize_total(clean, target_sum=1e4)
    sc.pp.log1p(clean)
    add_module_scores(clean)
    qc = {
        "sample": prefix,
        "age": age,
        "genotype": genotype,
        "raw_barcodes": raw.n_obs,
        "count_gene_QC": int(count_keep.sum()),
        "after_mt_QC": cells.n_obs,
        "EC_candidates": ec.n_obs,
        "mural_excluded": int((mural_n >= 2).sum()),
        "immune_excluded": int((immune_n >= 2).sum()),
        "clean_ECs": clean.n_obs,
    }
    del raw, candidate, cells, ec
    gc.collect()
    return clean, qc

zarkada, zarkada_qc_rows = {}, []
for prefix, age, genotype in ZARKADA_LIBRARIES:
    clean, qc = process_zarkada_library(prefix, age, genotype)
    zarkada[prefix] = clean
    zarkada_qc_rows.append(qc)
zarkada_qc = pd.DataFrame(zarkada_qc_rows)
expected_pre_state = {
    "GSM5350878_P6_WT": 296,
    "GSM5350880_P10_WT_D7": 110,
    "GSM5350881_P10_WT_C7": 333,
    "GSM5350879_P6_Alk5": 184,
    "GSM5350882_P10_Alk5_B7": 296,
    "GSM5350883_P10_Alk5_E7": 47,
}
assert zarkada_qc.set_index("sample")["clean_ECs"].to_dict() == expected_pre_state
zarkada_qc.to_csv(ZARKADA_OUT / "Zarkada_library_QC_summary.csv", index=False)
display(zarkada_qc)


In [ ]:
wt_names = [prefix for prefix, _, genotype in ZARKADA_LIBRARIES if genotype == "WT"]
zarkada_wt = ad.concat([zarkada[name] for name in wt_names], join="inner", merge="same", index_unique=None)
sc.pp.highly_variable_genes(zarkada_wt, n_top_genes=2000, flavor="seurat", batch_key="sample")
for gene in OUTCOME_GENES:
    if gene in zarkada_wt.var_names:
        zarkada_wt.var.loc[gene, "highly_variable"] = False
zarkada_wt_cluster = zarkada_wt[:, zarkada_wt.var["highly_variable"]].copy()
sc.pp.scale(zarkada_wt_cluster, max_value=10)
sc.tl.pca(zarkada_wt_cluster, n_comps=30, random_state=RANDOM_SEED)
sc.pp.neighbors(zarkada_wt_cluster, n_neighbors=15, n_pcs=20, random_state=RANDOM_SEED)
sc.tl.leiden(
    zarkada_wt_cluster,
    resolution=0.5,
    key_added="leiden_state",
    random_state=RANDOM_SEED,
    flavor="leidenalg",
    directed=True,
    n_iterations=-1,
)
zarkada_wt.obs["leiden_state"] = zarkada_wt_cluster.obs["leiden_state"].reindex(zarkada_wt.obs_names).astype(str)
expected_state_clusters = {"0": 213, "1": 164, "2": 142, "3": 100, "4": 62, "5": 58}
observed_state_clusters = zarkada_wt.obs["leiden_state"].value_counts().sort_index().to_dict()
assert observed_state_clusters == expected_state_clusters, (observed_state_clusters, expected_state_clusters)

state_marker_sets = {
    "general_tip": ["Mcam", "Chst1", "Nid2", "Rhoc", "Clec1a"],
    "S_tip": ["Esm1", "Angpt2", "Plvap"],
    "D_tip": ["Apod", "Pmepa1"],
    "cycling": ["Mki67", "Top2a", "Ccnb2"],
    "arterial_context": ["Gja5", "Bmx", "Efnb2", "Sox17"],
    "venous_context": ["Nr2f2", "Slc38a5"],
}
state_marker_genes = [gene for genes in state_marker_sets.values() for gene in genes if gene in zarkada_wt.var_names]
state_marker_rows = []
for cluster in sorted(zarkada_wt.obs["leiden_state"].unique(), key=int):
    subset = zarkada_wt[zarkada_wt.obs["leiden_state"].eq(cluster)]
    row = {"cluster": cluster, "n_cells": subset.n_obs}
    for gene in state_marker_genes:
        values = gene_vector(subset, gene)
        row[f"{gene}_mean"] = float(np.mean(values))
        row[f"{gene}_pct_positive"] = float(100 * np.mean(values > 0))
    state_marker_rows.append(row)
zarkada_state_marker_evidence = pd.DataFrame(state_marker_rows)
zarkada_state_marker_evidence.to_csv(ZARKADA_OUT / "Zarkada_WT_independent_state_marker_evidence.csv", index=False)

photo_genes = ["Rho", "Gnat1", "Pde6a", "Pde6b", "Pde6g", "Rcvrn", "Sag", "Gngt1", "Pdc", "Nr2e3", "Prph2"]
total_umi = np.asarray(zarkada_wt.layers["counts"].sum(axis=1)).ravel()
photo_counts = np.asarray(zarkada_wt[:, photo_genes].layers["counts"].sum(axis=1)).ravel()
photo_fraction = 100 * photo_counts / total_umi
cluster1 = zarkada_wt.obs["leiden_state"].astype(str).eq("1").to_numpy()
photo_audit = (
    pd.DataFrame({
        "cluster": zarkada_wt.obs["leiden_state"].astype(str).to_numpy(),
        "photoreceptor_pct_UMI": photo_fraction,
    })
    .groupby("cluster", observed=True)["photoreceptor_pct_UMI"]
    .agg(n_cells="size", median_pct_UMI="median", p90_pct_UMI=lambda values: np.percentile(values, 90))
    .reset_index()
)
photo_audit.to_csv(ZARKADA_OUT / "Zarkada_WT_cluster_photoreceptor_QC.csv", index=False)
photo_threshold = np.percentile(photo_fraction[~cluster1], 99)
high_photo_cluster1 = cluster1 & (photo_fraction > photo_threshold)
assert int(high_photo_cluster1.sum()) == 56
zarkada_wt_clean = zarkada_wt[~high_photo_cluster1].copy()
assert zarkada_wt_clean.n_obs == 683

ZARKADA_STATE_MAP = {
    "0": "D_tip_like",
    "1": "tip_intermediate",
    "2": "S_tip_like",
    "3": "venous_non_tip",
    "4": "arterial",
    "5": "cycling_EC",
}
zarkada_wt_clean.obs["state"] = zarkada_wt_clean.obs["leiden_state"].astype(str).map(ZARKADA_STATE_MAP)
expected_final_states = {"D_tip_like": 213, "S_tip_like": 142, "tip_intermediate": 108, "venous_non_tip": 100, "arterial": 62, "cycling_EC": 58}
assert zarkada_wt_clean.obs["state"].value_counts().to_dict() == expected_final_states
display(zarkada_state_marker_evidence)
display(photo_audit)
display(pd.crosstab(zarkada_wt_clean.obs["state"], zarkada_wt_clean.obs["sample"]))


In [ ]:
SCORE_COLUMNS = ["norrin_score", "barrier_score", "junction_score", "transport_score", "legacy_norrin_score"]

def tip_deltas(obs, state_column):
    rows = []
    for sample, frame in obs.groupby("sample", observed=True):
        s_tip = frame.loc[frame[state_column].eq("S_tip_like")]
        d_tip = frame.loc[frame[state_column].eq("D_tip_like")]
        if s_tip.empty or d_tip.empty:
            continue
        row = {"sample": sample, "age": str(frame["age"].iloc[0]), "n_S_tip": len(s_tip), "n_D_tip": len(d_tip)}
        for column in SCORE_COLUMNS:
            row[f"delta_D_minus_S_{column}"] = float(d_tip[column].median() - s_tip[column].median())
        rows.append(row)
    return pd.DataFrame(rows)

wt_tip_deltas = tip_deltas(zarkada_wt_clean.obs, "state")
wt_tip_deltas["analysis_arm"] = "WT_primary"

hvg_names = list(zarkada_wt_cluster.var_names)
wt_mean = np.array(zarkada_wt_cluster.var.loc[hvg_names, "mean"], dtype=float, copy=True)
wt_std = np.array(zarkada_wt_cluster.var.loc[hvg_names, "std"], dtype=float, copy=True)
wt_std[wt_std == 0] = 1.0
pc_loadings = np.asarray(zarkada_wt_cluster.varm["PCs"][:, :20])
train_mask = zarkada_wt_cluster.obs_names.isin(zarkada_wt_clean.obs_names)
train_names = zarkada_wt_cluster.obs_names[train_mask]
X_train = np.asarray(zarkada_wt_cluster.obsm["X_pca"][train_mask, :20])
y_train = zarkada_wt_clean.obs.loc[train_names, "state"].astype(str).to_numpy()
projector = KNeighborsClassifier(n_neighbors=15, weights="distance").fit(X_train, y_train)

alk5_names = [prefix for prefix, _, genotype in ZARKADA_LIBRARIES if genotype == "Alk5"]
projection_rows = []
for sample in alk5_names:
    obj = zarkada[sample]
    Xnew = obj[:, hvg_names].X
    Xnew = Xnew.toarray() if sparse.issparse(Xnew) else np.asarray(Xnew)
    Xnew_scaled = np.clip((Xnew - wt_mean) / wt_std, -10, 10)
    probabilities = projector.predict_proba(Xnew_scaled @ pc_loadings)
    best = np.argmax(probabilities, axis=1)
    confidence = probabilities[np.arange(len(best)), best]
    predicted = projector.classes_[best]
    obj.obs["projected_state"] = np.where(confidence >= 0.50, predicted, "unassigned")
    obj.obs["state_projection_confidence"] = confidence
    for state, frame in obj.obs.groupby("projected_state", observed=True):
        projection_rows.append({"sample": sample, "state": state, "n_cells": len(frame), "median_confidence": float(frame["state_projection_confidence"].median())})

alk5_obs = pd.concat([zarkada[name].obs for name in alk5_names], axis=0)
alk5_tip_deltas = tip_deltas(alk5_obs, "projected_state")
alk5_tip_deltas["analysis_arm"] = "Alk5_sensitivity"
zarkada_final_summary = pd.concat([wt_tip_deltas, alk5_tip_deltas], ignore_index=True)
ordered = ["sample", "age", "analysis_arm", "n_S_tip", "n_D_tip"] + [f"delta_D_minus_S_{column}" for column in SCORE_COLUMNS]
zarkada_final_summary = zarkada_final_summary[ordered]

assert int(zarkada_qc.loc[zarkada_qc["genotype"].eq("Alk5"), "clean_ECs"].sum()) == 527
assert len(zarkada_final_summary) == 6
for column in ["norrin_score", "barrier_score", "junction_score", "transport_score"]:
    assert (zarkada_final_summary[f"delta_D_minus_S_{column}"] > 0).all()
expected_norrin = np.array([0.120, 0.310, 0.150, 0.377, 0.256, 0.296])
expected_barrier = np.array([0.616, 0.643, 0.472, 0.654, 0.660, 0.721])
assert np.allclose(zarkada_final_summary["delta_D_minus_S_norrin_score"], expected_norrin, atol=0.0006)
assert np.allclose(zarkada_final_summary["delta_D_minus_S_barrier_score"], expected_barrier, atol=0.0006)

wt_tip_deltas.to_csv(ZARKADA_OUT / "Zarkada_WT_S_tip_D_tip_within_library_deltas.csv", index=False)
pd.DataFrame(projection_rows).to_csv(ZARKADA_OUT / "Zarkada_Alk5_projected_state_summary.csv", index=False)
zarkada_final_summary.to_csv(ZARKADA_OUT / "Zarkada_FINAL_6library_S_tip_D_tip_summary.csv", index=False)
display(zarkada_final_summary.round(3))


## Cross-dataset synthesis and figure

The datasets answer different questions and are not pooled into one effect estimate. The final table and figure juxtapose three complementary estimands: cross-cohort correlation, within-library AV-adjusted association, and independently defined state contrasts.


In [ ]:
cross_dataset_summary = pd.DataFrame([
    {
        "dataset": "MRCA",
        "analysis_unit": "11 sufficiently sampled EC sequencing cohorts",
        "n_cells": 1436,
        "primary_estimand": "Norrin-associated vs BRB module correlation",
        "primary_result": f"Random-effects pooled r = {primary['pooled_r']:.3f} (95% CI {primary['ci_low']:.3f}–{primary['ci_high']:.3f})",
        "direction": "positive",
        "major_caveat": f"Sequencing/preparation cohorts are not equivalent to biological replicates; I² = {primary['I2_percent']:.1f}%",
    },
    {
        "dataset": "Furtado GSE282775",
        "analysis_unit": "5 pooled genotype libraries",
        "n_cells": int(furtado_5library_summary["n_cells"].sum()),
        "primary_estimand": "Within-library Norrin–BRB Spearman association",
        "primary_result": "Positive in all 5 libraries; rho = 0.122–0.236",
        "direction": "positive",
        "major_caveat": "Cells are not biological replicates; genotype libraries are pooled; results are descriptive",
    },
    {
        "dataset": "Zarkada GSE175895",
        "analysis_unit": "3 WT libraries + 3 Alk5 sensitivity libraries",
        "n_cells": 1210,
        "primary_estimand": "Within-library D-tip-like minus S-tip-like module score",
        "primary_result": "D-tip-like > S-tip-like for Norrin, BRB, junction, and transport in 6/6 libraries",
        "direction": "positive",
        "major_caveat": "Descriptive state contrasts; unequal cell counts; smallest Alk5 library is supportive only",
    },
])
cross_dataset_summary.to_csv(FINAL_OUT / "cross_dataset_evidence_summary.csv", index=False)

mrca_plot = mrca_cohort_results.assign(r=mrca_cohort_results["r_primary"], n=mrca_cohort_results["n_EC"], label=mrca_cohort_results["sampleid"]).sort_values("r").reset_index(drop=True)
root_n = np.sqrt(mrca_plot["n"])
mrca_plot["point_size"] = 25 + 80 * (root_n - root_n.min()) / (root_n.max() - root_n.min())
furtado_plot = furtado_5library_summary.copy()
furtado_plot["label"] = furtado_plot["sample"].str.replace("GSM8650444_", "", regex=False).str.replace("GSM8650446_", "", regex=False).str.replace("GSM8650445_", "", regex=False).str.replace("GSM8650447_", "", regex=False).str.replace("GSM8650448_", "", regex=False)
zarkada_plot = zarkada_final_summary.copy()
zarkada_plot["label"] = ["P6 WT", "P10 WT D7", "P10 WT C7", "P6 Alk5", "P10 Alk5 B7", "P10 Alk5 E7"]

fig, axes = plt.subplots(1, 3, figsize=(17.5, 6.2), constrained_layout=True)
ax = axes[0]
y = np.arange(len(mrca_plot))
ax.scatter(mrca_plot["r"], y, s=mrca_plot["point_size"], zorder=3)
ax.axvline(0, color="0.35", linewidth=0.8)
ax.axvline(primary["pooled_r"], color="tab:blue", linestyle="--", linewidth=1.5)
ax.set(yticks=y, yticklabels=mrca_plot["label"], xlabel="Within-cohort Pearson r", xlim=(-0.25, 0.85), title="A  MRCA\nNorrin-associated vs BRB module")
ax.tick_params(axis="y", labelsize=7)
ax.text(0.03, 0.97, f"Random-effects r = {primary['pooled_r']:.3f}\n95% CI {primary['ci_low']:.3f}–{primary['ci_high']:.3f}\nI² = {primary['I2_percent']:.1f}%", transform=ax.transAxes, va="top", fontsize=9)
ax.text(0.03, 0.04, "Point size ∝ EC count", transform=ax.transAxes, fontsize=8)

ax = axes[1]
x = np.arange(len(furtado_plot))
ax.plot(x, furtado_plot["raw_Norrin_vs_BRB_rho"], marker="o", linewidth=1.5, label="Raw")
ax.plot(x, furtado_plot["AV_adjusted_partial_rho"], marker="s", linewidth=1.5, label="AV-adjusted")
ax.axhline(0, color="0.35", linewidth=0.8)
ax.set(xticks=x, xticklabels=furtado_plot["label"], ylabel="Within-library Spearman ρ", ylim=(-0.05, 0.30), title="B  Furtado\nNorrin–BRB association")
ax.tick_params(axis="x", rotation=40, labelsize=8)
ax.legend(frameon=False, fontsize=9, loc="upper left")

ax = axes[2]
x = np.arange(len(zarkada_plot)); width = 0.36
ax.bar(x - width / 2, zarkada_plot["delta_D_minus_S_norrin_score"], width, label="Norrin-associated")
ax.bar(x + width / 2, zarkada_plot["delta_D_minus_S_barrier_score"], width, label="BRB")
ax.axhline(0, color="0.35", linewidth=0.8); ax.axvline(2.5, color="0.35", linestyle=":", linewidth=1.2)
ax.set(xticks=x, xticklabels=zarkada_plot["label"], ylabel="Median D-tip-like − S-tip-like score", ylim=(-0.10, 0.80), title="C  Zarkada\nState-associated transcriptional shift")
ax.tick_params(axis="x", rotation=40, labelsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")

fig.suptitle("Three retinal endothelial datasets support coupling between Norrin-associated and BRB transcriptional programs", fontsize=14)
fig.savefig(FINAL_OUT / "figure_computational_triangulation.png", dpi=600, bbox_inches="tight")
fig.savefig(FINAL_OUT / "figure_computational_triangulation.pdf", bbox_inches="tight")
plt.show()
display(cross_dataset_summary)


In [ ]:
input_paths = [mrca_path]
for prefix, _, _ in FURTADO_LIBRARIES:
    input_paths.extend(_find_triplet_file(FURTADO_DIR, prefix, kind) for kind in ["barcodes", "features", "matrix"])
for prefix, _, _ in ZARKADA_LIBRARIES:
    input_paths.extend(_find_triplet_file(ZARKADA_DIR, prefix, kind) for kind in ["barcodes", "features", "matrix"])
input_manifest = write_input_manifest(input_paths)
run_summary = {
    "status": "completed",
    "modules": MODULES,
    "random_seed": RANDOM_SEED,
    "mrca": primary,
    "furtado_clean_ecs": int(furtado_5library_summary["n_cells"].sum()),
    "zarkada_wt_clean_ecs": int(zarkada_wt_clean.n_obs),
    "zarkada_alk5_clean_ecs": int(zarkada_qc.loc[zarkada_qc["genotype"].eq("Alk5"), "clean_ECs"].sum()),
    "zarkada_directionally_positive_libraries": 6,
    "caveat": "Cell-level analyses are descriptive; cells are not independent biological replicates.",
}
(FINAL_OUT / "run_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print("All fail-fast validation checks passed.")
print("Generated outputs:", FINAL_OUT)
display(input_manifest)


## Interpretation boundary

The analyses support a modest, context-dependent association between Norrin-associated endothelial identity and BRB-associated transcriptional identity. They do not demonstrate direct Norrin pathway activity, causality, a validated biomarker, or genotype-level effects. Shared developmental state and other endothelial maturation programs remain plausible explanations. The MRCA heterogeneity (`I² ≈ 66.5%`) is a result to report, not noise to hide.
